In [1]:
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import pandas as pd
import textwrap
from simtag import covariate_labeling, knn_comparison, count_df, plot_df, adjust_tag, adjust_distribution
from tqdm import tqdm
tqdm.pandas()

# we instatiate the model first, so we can assign it to multiple copies of the engine
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# import raw data
df = pd.read_parquet('notebooks/games.parquet').dropna()
df['Tags'] = df['Tags'].apply(lambda x : x.split(','))
df['Genres'] = df['Genres'].apply(lambda x : x.split(','))
df = df.drop(['game_vector', 'game_indices', 'Score', 'Recommendations'], axis=1)

# extract raw lists
sample_list = df['Tags'].values.tolist()
tag_list = sorted(list(set([x for xs in sample_list for x in xs])))
tag_encoded = [model.encode(x) for x in tag_list]

index = NearestNeighbors(n_neighbors=20, metric='cosine').fit(tag_encoded)

In [3]:
sample = df.sample(1)['About the game']
str1 = sample.tolist()[0]
i = sample.index.tolist()[0]

vector = model.encode(str1)
values, tags_top, scores = covariate_labeling(tag_list, tag_encoded, vector, inflection_point=0)

print('index:', i)
print(textwrap.fill(str1, width=120), '\n')

knn_tags = knn_comparison(index, vector, tag_list, tags_top)

print('TAGS:\t\t', tags_top)
print('knn_comparison\t', knn_tags)

index: 22539
Magic Trick is a magical skateboarding adventure game, featuring expressive skating movement and trick combos, a totally
explorable whimsical sunset world, and adorable townsfolk offering quests and companionship. Magic Trick seeks to
fulfill everyone’s greatest desire: being a skateboarding magician. The player embodies Sunset Isles’ newest resident:
WizKid (pronouns: they/them), who can perform radical magic by using skate trick combos! The day is nearly over, but
Wizkid wants to explore the town and meet their new neighbors before calling it a day. The twilight adventure begins at
the skate park to hone their skills with skateboarding legend, Tony the Hawk. Tony teaches the player skating tricks
ranging from a simple ollie to a gnarly backflip, while Wizkid’s magic allows the player to perform incredible feats
such as skating at super speed, spawning a ramp from thin air, and flying to the top of the sky. The world is the
player’s skatepark, with ramps and grind rails r

### tag the whole dataset

In [4]:
# tag the whole dataset
df_ = df.sample(100)
df_['_tags'] = df_['About the game'].progress_apply(lambda x : covariate_labeling(
    tag_list, tag_encoded, model.encode(x), inflection_point=0)[2]
)
df_['_id'] = df_.index

100%|██████████| 100/100 [00:10<00:00,  9.53it/s]


### adjust tag distribution

In [7]:
count_df(df_)[0:10]

[[14, 'Choose Your Own Adventure'],
 [12, 'Open World Survival Craft'],
 [10, 'Gun Customization'],
 [9, 'Villain Protagonist'],
 [9, 'Turn-Based Tactics'],
 [9, 'Female Protagonist'],
 [8, 'Mystery Dungeon'],
 [7, 'Turn-Based Combat'],
 [7, 'Roguelike Deckbuilder'],
 [7, 'Puzzle-Platformer']]

In [21]:
df_new = adjust_distribution(df_.copy(), max_tag_density=0.04, max_iter=10)
count_df(df_new)[0:10]

[[7, 'Mystery Dungeon'],
 [7, 'Music-Based Procedural Generation'],
 [7, 'Level Editor'],
 [7, 'Job Simulator'],
 [7, 'Escape Room'],
 [7, 'Character Action Game'],
 [6, 'Turn-Based Strategy'],
 [6, 'Turn-Based Combat'],
 [6, 'Tower Defense'],
 [6, 'Tactical RPG']]